# PHYCOM R1 — full Kaggle T4 ×2 experiment notebook

**Manuscript:** PHYCOM-D-26-02646, submitted 16 September 2026.

This notebook contains the complete V4 source. It unpacks the Python files locally;
no GitHub download, input dataset, Mamba build or pip change is needed for the first run.
The separate source ZIP contains readable source and an unchanged V3.2 reference copy.

The new experiments address projection/LSTM bottlenecks, ungated raw LSTM,
matched initialization, independent seeds, correlated channels, estimated-statistics
LMMSE, LS residual corrections, spectral diagnostics and in/out-of-range SNR errors.
The earlier Mamba/GRU experiments remain a separate V3.2 study.

**Before Run all:** select **GPU T4 x2** in Kaggle's accelerator settings. The full
plan has **430 fits**, with ten central seeds and five supplementary seeds.
It can require multiple sessions. Set the budget below to fit your remaining
session time and quota, allowing at least an hour for exports. The notebook does
not guarantee a speedup factor, an accuracy gain, or journal acceptance.

Local validation used CPU, including a complete 108-fit tiny smoke run. Real CUDA
forward/backward checks run on each GPU here before training. Smoke results are
not publication evidence. All final model choices are frozen before confirmation.

## 1. Settings

Keep `SMOKE_TEST = False` for the research experiments. Optional `True` runs a tiny,
two-epoch pipeline check in a separate directory. When resuming, attach the last
**RESUME.zip** as a private Kaggle input. `auto` finds one archive; with several
inputs specify the exact archive path. Keep the same source and numerical runtime.

In [ ]:
SMOKE_TEST = False
SESSION_HOURS = 10.0  # Work budget. Reduce to match YOUR remaining session time/quota.
USE_BOTH_GPUS = True
RESUME_INPUT = "auto"  # First session: no input required. Later: attach RESUME.zip.

from pathlib import Path
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
CODE_DIR = WORK / "PHYCOM_V4_CODE"
OUT_DIR = WORK / ("PHYCOM_V4_SMOKE_RUN" if SMOKE_TEST else "PHYCOM_V4_RUN")

## 2. Unpack the complete, checksummed source

The embedded archive is just the Python code, documentation, and validation notes.
You can inspect the extracted files in `PHYCOM_V4_CODE`. Existing changed source
files are not silently overwritten. Changing scientific code requires a fresh run.

In [ ]:
import base64, hashlib, io, zipfile
SOURCE_SHA256 = 'caf9ac643229407ea31c6d0b2ae2018090347853a38cadcff1ac87df03b0824e'
SOURCE_BASE64 = (
    '...'
    'BgAAAAAeAB4AQggAABjWAAAAAA=='
)
raw = base64.b64decode(SOURCE_BASE64)
assert hashlib.sha256(raw).hexdigest() == SOURCE_SHA256, "Embedded source checksum failed"
CODE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for info in z.infolist():
        target = (CODE_DIR / info.filename).resolve()
        if not target.is_relative_to(CODE_DIR.resolve()):
            raise ValueError("Unsafe source archive path")
        data = z.read(info)
        if target.exists() and target.read_bytes() != data:
            raise RuntimeError(f"Existing source differs: {target}. Keep it as a separate version.")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
print("Complete source extracted to", CODE_DIR)
print((CODE_DIR / "REVIEWER_MAP.md").read_text()[:600])

## 3. Check hardware and dependencies

The runner makes two independent GPU workers, preserving the matched batch size.
No mixed-precision or distributed-training changes are introduced. A single GPU
is supported only if you deliberately set `USE_BOTH_GPUS = False`.

In [ ]:
import os, sys, subprocess
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch, numpy, scipy, pandas, matplotlib
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
for index in range(torch.cuda.device_count()):
    print("GPU", index, torch.cuda.get_device_name(index))
if not torch.cuda.is_available():
    if not SMOKE_TEST:
        raise RuntimeError("Enable GPU T4 x2 in Kaggle settings and restart the session.")
    DEVICES = "cpu"
elif USE_BOTH_GPUS:
    if torch.cuda.device_count() < 2:
        raise RuntimeError("Two GPUs were requested. Select GPU T4 x2, or deliberately set USE_BOTH_GPUS=False.")
    DEVICES = "0,1"
else:
    DEVICES = "0"
assert SESSION_HOURS > 0, "Choose a positive session work budget"
print("Workers:", DEVICES, "Output:", OUT_DIR)

## 4. Train, freeze selections, evaluate and export

Leave this cell running. Progress reports show the group assigned to each GPU;
individual epoch logs are in the output `logs` directory. Epoch checkpoints are
atomic. Once the time budget is reached, committed progress is packaged for the
next session. Evaluation starts after the whole study's model selection is frozen.

**Let archive creation finish before stopping the session.** Keep both generated
ZIPs. A hard Kaggle timeout before export can prevent creation of the final ZIP.

In [ ]:
cmd = [sys.executable, "-u", str(CODE_DIR / "run_revision.py"),
       "--output", str(OUT_DIR), "--devices", DEVICES,
       "--hours", str(SESSION_HOURS), "--resume-input", RESUME_INPUT]
if SMOKE_TEST:
    cmd.append("--smoke")
subprocess.run(cmd, cwd=str(CODE_DIR), check=True)

## 5. Download and resume

- **RESUME.zip:** save after every session; attach it to the next session as a
  private input. It contains selected models, optimizer state, raw errors and source.
- **REVIEW.zip:** upload this here when the full run is complete so that the paper
  and point-by-point reviewer responses can be revised from actual evidence.
- `complete: true` confirms scheduled tasks finished. Check convergence flags
  before scientific interpretation. Tiny `SMOKE` outputs are never paper results.
- The Files/Output panel also provides the ZIPs if the links below do not download.

Do not edit seeds, model widths or training settings midway through a run. Resume
with the same notebook and numerical package versions; different run identities
are deliberately rejected rather than silently mixed.

In [ ]:
import json
from IPython.display import display, FileLink, Markdown
for name in ["SESSION_STATUS.json", "FINAL_AUDIT.json"]:
    p = OUT_DIR / name
    if p.exists():
        print(name)
        print(json.dumps(json.loads(p.read_text()), indent=2))
mode = "SMOKE" if SMOKE_TEST else "PUBLICATION"
for name in [f"PHYCOM_V4_{mode}_REVIEW.zip", f"PHYCOM_V4_{mode}_RESUME.zip"]:
    p = WORK / name
    if p.exists():
        print(name, f"({p.stat().st_size / 1024**2:.1f} MiB)")
        display(FileLink(str(p)))
summary = OUT_DIR / "Paper_Data" / "RESULTS_SUMMARY.md"
if summary.exists():
    display(Markdown(summary.read_text()))

## Protocol and source

Read `PHYCOM_V4_CODE/PROTOCOL.md`, `REVIEWER_MAP.md`, and `TEST_REPORT.md` for the
design, reviewer mapping and validation limits. `RUN_MANIFEST.json` identifies the
actual numerical runtime. Numerical and environment changes can affect
reproducibility; see the official [PyTorch reproducibility notes](https://docs.pytorch.org/docs/stable/notes/randomness.html).
Kaggle controls and resource availability are documented in the official
[notebook documentation](https://www.kaggle.com/docs/notebooks).